In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import torch

# Paths where trained models are saved in Google Drive
DEBERTA_PATH = "/content/drive/My Drive/DeBERTa_model"
ROBERTA_PATH = "/content/drive/MyDrive/RoBERTa_Model"
T5_PATH      = "/content/drive/MyDrive/T5_Model"

In [ ]:
!unzip multilabel-classification-dataset.zip -d /content/dataset

In [ ]:
import pandas as pd

# Load your CSV
df = pd.read_csv("/content/dataset/train.csv")

# Define the label columns
label_columns = [
    'Computer Science',
    'Physics',
    'Mathematics',
    'Statistics',
    'Quantitative Biology',
    'Quantitative Finance'
]

# Count how many samples belong to each class (sum since multilabel has 0/1 per class)
class_counts = df[label_columns].sum().astype(int)

print(" Class distribution (number of samples per label):")
print(class_counts)


In [ ]:
# Combine TITLE + ABSTRACT like in training
df["text"] = df["TITLE"].astype(str) + " " + df["ABSTRACT"].astype(str)

LABEL_NAMES = [
    "Computer Science",
    "Physics",
    "Mathematics",
    "Statistics",
    "Quantitative Biology",
    "Quantitative Finance"
]

texts = df["text"].tolist()
labels = df[LABEL_NAMES].values.astype(int)

print("Sample text:", texts[0][:400])
print("Labels:", labels[0])

In [ ]:
!pip install scikit-multilearn


In [ ]:
from skmultilearn.model_selection import iterative_train_test_split


In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/dataset/train.csv")

LABEL_NAMES = ["Computer Science","Physics","Mathematics","Statistics","Quantitative Biology","Quantitative Finance"]

# Merge TITLE + ABSTRACT
texts = (df["TITLE"] + " " + df["ABSTRACT"]).tolist()
labels = df[LABEL_NAMES].values  # shape (N,6)

# Redo split if needed
from skmultilearn.model_selection import iterative_train_test_split
X = np.array(texts).reshape(-1, 1)
y = labels
X_train, y_train, X_val, y_val = iterative_train_test_split(X, y, test_size=0.15)

train_texts, train_labels = X_train.ravel(), y_train
val_texts, val_labels = X_val.ravel(), y_val


In [ ]:
# Step 1: Sanity checks
import torch, numpy as np
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Check model & tokenizer presence
required = ["deberta_model","deberta_tokenizer","roberta_model","roberta_tokenizer","t5_model","t5_tokenizer","val_texts","val_labels"]
for name in required:
    print(f"{name}: {'FOUND' if name in globals() else 'MISSING'}")

# Quick checks for val_texts / val_labels
try:
    print("Number of validation samples:", len(val_texts))
    import numpy as np
    arr = np.array(val_labels)
    print("val_labels shape:", arr.shape)
    print("Sample text (first):", val_texts[0][:400])
except Exception as e:
    raise RuntimeError("val_texts / val_labels not set correctly. Make sure val_texts is a list[str] and val_labels is numpy array of shape (N,6). Error: "+str(e))

# Define LABEL_NAMES if not present
if "LABEL_NAMES" not in globals():
    LABEL_NAMES = ["Computer Science","Physics","Mathematics","Statistics","Quantitative Biology","Quantitative Finance"]
    print("LABEL_NAMES not found — default set.")
else:
    print("LABEL_NAMES found.")


In [ ]:
# Step 2: prediction helper (supports classifier logits or generation fallback)
import numpy as np
import torch

def predict_probs(model, tokenizer, texts, batch_size=8, max_length=512, label_names=None):
    """
    Returns numpy array shape (N, C) of probabilities in [0,1].
    - If model is a classifier, uses sigmoid(logits).
    - If model is generation-only, generates text and maps label names presence -> 0/1.
    """
    if label_names is None:
        label_names = LABEL_NAMES
    model.to(DEVICE)
    model.eval()
    all_probs = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        enc = {k:v.to(DEVICE) for k,v in enc.items()}

        with torch.no_grad():
            try:
                out = model(**enc)
                logits = out.logits  # classifier: (B, C) ; generation: (B, seq_len, vocab)
            except Exception as e:
                # Some models require different call signatures
                out = model(input_ids=enc["input_ids"], attention_mask=enc.get("attention_mask", None))
                logits = out.logits

            # If logits are 3D, treat as generation-ish (vocab dim)
            if logits is None:
                raise RuntimeError("Model.forward did not return logits. Model type unexpected.")
            if logits.ndim == 3 and logits.shape[-1] > 1000:
                # generation fallback: produce text and parse
                generated = model.generate(**enc, max_length=64)
                gen_texts = [tokenizer.decode(g, skip_special_tokens=True).lower() for g in generated]
                batch_probs = []
                for gt in gen_texts:
                    row = [1.0 if lbl.lower() in gt else 0.0 for lbl in label_names]
                    batch_probs.append(row)
                batch_probs = np.array(batch_probs, dtype=float)
            else:
                # classifier path (B, C)
                batch_probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(batch_probs)
        torch.cuda.empty_cache()

    return np.vstack(all_probs)


In [ ]:
# Ensure val_texts is a list of strings
if not isinstance(val_texts, list):
    val_texts = val_texts.ravel().tolist()
else:
    val_texts = [str(x) for x in val_texts]

print("First 2 validation texts:\n", val_texts[:2])
print("Number of samples:", len(val_texts))


In [ ]:
from transformers import T5ForSequenceClassification

t5_model = T5ForSequenceClassification.from_pretrained("/content/drive/My Drive/T5_Model")


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load DeBERTa model and tokenizer
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH, local_files_only=True)
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH, local_files_only=True)

In [ ]:
# Load DeBERTa model and tokenizer
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH, local_files_only=True)
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_PATH, local_files_only=True)

In [ ]:
# Load DeBERTa model and tokenizer
t5_model = AutoModelForSequenceClassification.from_pretrained(T5_PATH, local_files_only=True)
t5_tokenizer = AutoTokenizer.from_pretrained(T5_PATH, local_files_only=True)

In [ ]:
# Step 3: run inference (will print shapes)
print("Running DeBERTa inference...")
probs_deberta = predict_probs(deberta_model, deberta_tokenizer, val_texts, batch_size=8, max_length=512)
print("DeBERTa probs shape:", probs_deberta.shape)

print("Running RoBERTa inference...")
probs_roberta = predict_probs(roberta_model, roberta_tokenizer, val_texts, batch_size=8, max_length=512)
print("RoBERTa probs shape:", probs_roberta.shape)

print("Running T5 inference...")
probs_t5 = predict_probs(t5_model, t5_tokenizer, val_texts, batch_size=8, max_length=512)  # smaller batch if T5 heavy
print("T5 probs shape:", probs_t5.shape)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#  Step 4: Simple Average Ensemble
probs_ensemble = (probs_deberta + probs_roberta + probs_t5) / 3

# Apply threshold
preds_ensemble = (probs_ensemble >= 0.5).astype(int)

#  Metrics
accuracy = accuracy_score(val_labels, preds_ensemble)
precision = precision_score(val_labels, preds_ensemble, average="micro", zero_division=0)
recall = recall_score(val_labels, preds_ensemble, average="micro", zero_division=0)
f1 = f1_score(val_labels, preds_ensemble, average="micro", zero_division=0)

#  Ensemble loss (validation)
criterion = nn.BCEWithLogitsLoss()
eps = 1e-8
ensemble_logits = torch.log(torch.tensor(probs_ensemble + eps) / (1 - torch.tensor(probs_ensemble) + eps))
labels_tensor = torch.tensor(val_labels, dtype=torch.float32)
ensemble_loss = criterion(ensemble_logits, labels_tensor).item()

print("🔹 Ensemble Results (Simple Average)")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Loss:      {ensemble_loss:.4f}")

# Per-label F1
per_label_f1 = f1_score(val_labels, preds_ensemble, average=None, zero_division=0)
print("\nPer-label F1:", per_label_f1)
print("Mean per-label F1:", per_label_f1.mean())


In [ ]:
# Apply threshold (e.g., 0.5 by default)
threshold = 0.5
y_pred_ensemble = (probs_ensemble >= threshold).astype(int)

print("y_pred_ensemble shape:", y_pred_ensemble.shape)


In [ ]:
from sklearn.metrics import accuracy_score

# Subset accuracy (exact match: all labels correct for a sample)
subset_acc = accuracy_score(val_labels, y_pred_ensemble)

# Per-label accuracy
per_label_acc = (val_labels == y_pred_ensemble).mean(axis=0)

# Mean per-label accuracy
mean_per_label_acc = per_label_acc.mean()

print("Subset Accuracy (exact match):", subset_acc)
print("\nPer-label Accuracy:")
for i, acc in enumerate(per_label_acc):
    print(f"  {LABEL_NAMES[i]}: {acc:.4f}")

print("\nMean Per-label Accuracy:", mean_per_label_acc)


In [ ]:
subset_acc = accuracy_score(val_labels, y_pred_ensemble)
print("Subset Accuracy (exact match):", subset_acc)


In [ ]:
per_label_acc = (val_labels == y_pred_ensemble).mean(axis=0)
mean_per_label_acc = per_label_acc.mean()

print("\nPer-label Accuracy:")
for i, acc in enumerate(per_label_acc):
    print(f"  {LABEL_NAMES[i]}: {acc:.4f}")
print("\nMean Per-label Accuracy:", mean_per_label_acc)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

print("🔹 Ensemble Results")

# --- Micro metrics (treat all labels equally across samples) ---
micro_acc = accuracy_score(val_labels.flatten(), y_pred_ensemble.flatten())
micro_prec = precision_score(val_labels, y_pred_ensemble, average="micro", zero_division=0)
micro_rec  = recall_score(val_labels, y_pred_ensemble, average="micro", zero_division=0)
micro_f1   = f1_score(val_labels, y_pred_ensemble, average="micro", zero_division=0)

print(f"Accuracy (micro over all labels) = {micro_acc:.4f}")
print(f"Precision (micro-average) = {micro_prec:.4f}")
print(f"Recall (micro-average) = {micro_rec:.4f}")
print(f"F1 Score (micro-average) = {micro_f1:.4f}\n")

# --- Per-label accuracy ---
print("Per-label Accuracy:")
per_label_acc = []
for i, label in enumerate(LABEL_NAMES):
    acc = accuracy_score(val_labels[:, i], y_pred_ensemble[:, i])
    per_label_acc.append(acc)
    print(f"({label}, {acc:.6f})")

# --- Average accuracy across labels ---
mean_acc = np.mean(per_label_acc)
print("\nAverage accuracy = ", mean_acc)


In [ ]:
import joblib
import os

# Define where to save in Google Drive
SAVE_PATH = "/content/drive/MyDrive/Ensemble_Model_Updated"
os.makedirs(SAVE_PATH, exist_ok=True)

# Save thresholds (if you tuned them)
thresholds = [0.5] * len(LABEL_NAMES)  # replace if you found custom thresholds
joblib.dump(thresholds, os.path.join(SAVE_PATH, "ensemble_thresholds.pkl"))

# Save ensemble metadata (which models, how combined)
ensemble_config = {
    "models": {
        "deberta": DEBERTA_PATH,
        "roberta": ROBERTA_PATH,
        "t5": T5_PATH,
    },
    "method": "simple_average",
    "thresholds": thresholds,
    "labels": LABEL_NAMES,
}
joblib.dump(ensemble_config, os.path.join(SAVE_PATH, "ensemble_config.pkl"))

print(" Ensemble config and thresholds saved at:", SAVE_PATH)


In [ ]:
def ensemble_predict(texts, max_length=512, batch_size=8):
    # Get probs from each model
    p1 = predict_probs(deberta_model, deberta_tokenizer, texts, batch_size, max_length)
    p2 = predict_probs(roberta_model, roberta_tokenizer, texts, batch_size, max_length)
    p3 = predict_probs(t5_model, t5_tokenizer, texts, batch_size, max_length)

    # Simple average ensemble
    probs = (p1 + p2 + p3) / 3

    # Apply thresholds
    preds = (probs >= np.array(thresholds)).astype(int)
    return preds, probs


In [ ]:
# Get predictions from each base model
p1 = predict_probs(deberta_model, deberta_tokenizer, val_texts, batch_size=8, max_length=512)
p2 = predict_probs(roberta_model, roberta_tokenizer, val_texts, batch_size=8, max_length=512)
p3 = predict_probs(t5_model, t5_tokenizer, val_texts, batch_size=8, max_length=512)

# Simple average ensemble
probs = (p1 + p2 + p3) / 3

# Apply thresholds (default 0.5 per label, or your tuned ones)
thresholds = [0.5] * probs.shape[1]
preds = (probs >= np.array(thresholds)).astype(int)

print(" Ensemble predictions ready. Shape:", preds.shape)


In [ ]:
np.save(os.path.join(SAVE_PATH, "val_probs_ensemble.npy"), probs)
np.save(os.path.join(SAVE_PATH, "val_preds_ensemble.npy"), preds)

print(" Saved ensemble predictions at:", SAVE_PATH)
